# Image Enhancement and Per-Image Selection (Phase 0)
**Team Triathlon**: Ananya Kasaudhan, Suyash Srivastav, Logi Vasan  
**Supervisor**: Prof. Kishor Upla, Department of Electronics Engineering, SVNIT Surat  
---
### Overview
This notebook explores per-image enhancement selection formulated as an algorithm selection problem (Rice, 1976).
We use 7 cheap handcrafted physical statistics from degraded images to predict the PSNR-optimal enhancer from a 6-method bank.

In [ ]:
import sys
sys.path.append("..")
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

from src.methods import METHODS
from src.features import extract_features, FEATURE_NAMES
from src.labels import compute_psnr, compute_ssim, score_pair
from src.dataset import load_and_preprocess_pair

print("Libraries and project modules successfully loaded!")


## 1. Inspecting the Six Enhancement Methods on an Example Image

In [ ]:
low_path = "../data/lol_dataset/eval15/low/1.png"
high_path = "../data/lol_dataset/eval15/high/1.png"
clean, deg, scale = load_and_preprocess_pair(low_path, high_path, target_long_side=512)

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

axes[0].imshow(cv2.cvtColor(deg, cv2.COLOR_BGR2RGB))
axes[0].set_title("Degraded Input (Low-Light)")
axes[0].axis("off")

axes[1].imshow(cv2.cvtColor(clean, cv2.COLOR_BGR2RGB))
axes[1].set_title("Ground Truth Reference")
axes[1].axis("off")

for idx, (m_name, m_fn) in enumerate(METHODS.items()):
    out = m_fn(deg)
    p = compute_psnr(clean, out)
    s = compute_ssim(clean, out)
    axes[idx + 2].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    axes[idx + 2].set_title(f"{m_name} (PSNR: {p:.2f} dB | SSIM: {s:.3f})")
    axes[idx + 2].axis("off")

plt.tight_layout()
plt.show()


## 2. Extracting Handcrafted Physical Features

In [ ]:
feats = extract_features(deg, include_engineered=True)
df_f = pd.DataFrame([feats])
print("Degraded Image Handcrafted Physical Descriptors:")
display(df_f.T.rename(columns={0: "Value"}))

## 3. Loading Benchmark Results & Regret Tables

In [ ]:
df_summary = pd.read_csv("../results/metrics_table.csv")
print("=== Primary Benchmark Results (GroupKFold CV) ===")
display(df_summary)

df_eval15 = pd.read_csv("../results/eval15_test_results.csv")
print("=== Official eval15 Test Results ===")
display(df_eval15)

## 4. Visualizing Model Evaluation Figures

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 13))

img1 = cv2.imread("../results/figures/fig1_oracle_distribution.png")
axes[0, 0].imshow(cv2.cvtColor(img1, cv2.COLOR_BGR2RGB))
axes[0, 0].set_title("Figure 1: Oracle Method Distribution")
axes[0, 0].axis("off")

img2 = cv2.imread("../results/figures/fig2_confusion_matrix.png")
axes[0, 1].imshow(cv2.cvtColor(img2, cv2.COLOR_BGR2RGB))
axes[0, 1].set_title("Figure 2: Confusion Matrix (Random Forest)")
axes[0, 1].axis("off")

img3 = cv2.imread("../results/figures/fig3_regret_cdf.png")
axes[1, 0].imshow(cv2.cvtColor(img3, cv2.COLOR_BGR2RGB))
axes[1, 0].set_title("Figure 3: Empirical CDF of Regret (dB)")
axes[1, 0].axis("off")

img4 = cv2.imread("../results/figures/fig4_feature_importances.png")
axes[1, 1].imshow(cv2.cvtColor(img4, cv2.COLOR_BGR2RGB))
axes[1, 1].set_title("Figure 4: Feature Importances")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Statistical Significance Testing
Paired two-tailed t-tests, Wilcoxon signed-rank tests, and 95% confidence intervals across all 15 grouped cross-validation folds:

In [ ]:
df_stats = pd.read_csv("../results/statistical_tests.csv")
print("=== Statistical Significance Testing (RF vs. Baselines across 15 Runs) ===")
display(df_stats)

## 6. Qualitative Restoration Comparison Strip (Figure 5)
Contrasts degraded inputs, the best fixed method (Retinex), the proposed selector output, and ground truth across diverse scenes:

In [ ]:
plt.figure(figsize=(18, 12))
img5 = cv2.imread("../results/figures/fig5_qualitative_strip.png")
plt.imshow(cv2.cvtColor(img5, cv2.COLOR_BGR2RGB))
plt.title("Figure 5: Qualitative Restoration Comparison Strip (LOL v1 Benchmark)", fontsize=14, fontweight="bold")
plt.axis("off")
plt.show()

## 7. Feature Ablation Study
Impact of removing individual feature groups on Top-1 Accuracy and Decision Regret:

In [ ]:
df_ablation = pd.read_csv("../results/ablation_table.csv")
display(df_ablation)

## 8. Cross-Dataset Generalization: LOL-v2 (Synthetic & Real)
To test robustness against domain shift and cross-sensor hardware variations, we evaluate zero-shot transfer on **LOL-v2 Synthetic** (physical Poisson degradation model) and **LOL-v2 Real** (dual-camera cross-sensor pairs).

In [ ]:
df_v2_syn = pd.read_csv("../results/lol_v2_generalization_synthetic.csv")
df_v2_real = pd.read_csv("../results/lol_v2_generalization_real.csv")

print("=== LOL-v2 Synthetic Zero-Shot Transfer ===")
display(df_v2_syn)

print("=== LOL-v2 Real Cross-Camera Transfer ===")
display(df_v2_real)

# Plot cross-dataset generalization comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
splits = ["LOL-v1 (CV)", "LOL-v2 Syn (Zero-Shot)", "LOL-v2 Real (Cross-Cam)"]
accuracies = [67.42, float(df_v2_syn.loc[df_v2_syn["Strategy"].str.contains("Proposed"), "Top-1 Accuracy (%)"].iloc[0]), float(df_v2_real.loc[df_v2_real["Strategy"].str.contains("Proposed"), "Top-1 Accuracy (%)"].iloc[0])]
regrets = [1.16, float(df_v2_syn.loc[df_v2_syn["Strategy"].str.contains("Proposed"), "Mean Regret (dB)"].iloc[0]), float(df_v2_real.loc[df_v2_real["Strategy"].str.contains("Proposed"), "Mean Regret (dB)"].iloc[0])]

ax1.bar(splits, accuracies, color=["#2563eb", "#10b981", "#f59e0b"], width=0.5)
ax1.set_ylabel("Top-1 Accuracy (%)", fontsize=11, fontweight="bold")
ax1.set_title("Top-1 Accuracy Across Datasets", fontsize=12, fontweight="bold")
ax1.set_ylim(0, 100)
for i, v in enumerate(accuracies):
    ax1.text(i, v + 2, f"{v:.1f}%", ha="center", fontweight="bold")

ax2.bar(splits, regrets, color=["#2563eb", "#10b981", "#f59e0b"], width=0.5)
ax2.set_ylabel("Mean Regret (dB)", fontsize=11, fontweight="bold")
ax2.set_title("Mean Decision Regret Across Datasets", fontsize=12, fontweight="bold")
ax2.set_ylim(0, max(regrets) + 1.5)
for i, v in enumerate(regrets):
    ax2.text(i, v + 0.15, f"{v:.2f} dB", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 9. Downstream Vision Benchmark: YOLOv8 Object Detection
Enhancement operators must preserve semantics for machine vision. Here we evaluate bounding-box recall with YOLOv8n on the LOL test set.

In [ ]:
df_yolo = pd.read_csv("../results/downstream_yolo_metrics.csv")
print("Downstream YOLOv8 Detection Per-Scene Metrics:")
display(df_yolo.head(10))

total_raw = df_yolo["raw_boxes"].sum()
total_ret = df_yolo["retinex_boxes"].sum()
total_sel = df_yolo["selector_boxes"].sum()
total_ref = df_yolo["ref_boxes"].sum()

mean_rec_raw = df_yolo["recall_raw_vs_ref"].mean()
mean_rec_ret = df_yolo["recall_retinex_vs_ref"].mean()
mean_rec_sel = df_yolo["recall_selector_vs_ref"].mean()

print(f"Total Bounding Boxes Detected (Test Set): Raw={total_raw}, Retinex={total_ret}, Selector={total_sel}, GT Reference={total_ref}")
print(f"Relative Recall vs Reference: Raw={mean_rec_raw:.1f}%, Retinex={mean_rec_ret:.1f}%, Proposed Selector={mean_rec_sel:.1f}%")

fig, ax = plt.subplots(figsize=(8, 4.5))
methods = ["Degraded Input", "Fixed Retinex", "Proposed Selector", "GT Reference"]
recalls = [mean_rec_raw, mean_rec_ret, mean_rec_sel, 100.0]
colors = ["#94a3b8", "#ef4444", "#22c55e", "#3b82f6"]
bars = ax.bar(methods, recalls, color=colors, width=0.5)
ax.set_ylabel("Relative Bounding Box Recall (%)", fontsize=11, fontweight="bold")
ax.set_title("Downstream YOLOv8 Object Detection Benchmark", fontsize=12, fontweight="bold")
ax.set_ylim(0, 115)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 2, f"{yval:.1f}%", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 10. Edge Hardware & Latency Profiling (ONNX Runtime)
In-depth hardware comparison evaluating single-core CPU latency, throughput, and memory consumption against deep learning architectures.

In [ ]:
df_edge = pd.read_csv("../results/edge_hardware_profile.csv")
display(df_edge)

print("Key Finding: Proposed Selector achieves 11.81 ms inference (84.7 FPS) on CPU with only 54.4 MB peak RAM, requiring ZERO GPU acceleration.")